# Phase 0 — go / no-go before any spend

Four measurements, all on code and data that already exist. $0, no retrieval, no API calls.
Plan: `.claude/plans/reframe-the-multi-leg-tie-resolution-foamy-dawn.md`.

**This notebook is a hard gate.** Per the plan's sequencing: nothing is bought (no re-retrieval,
no reranker run, no LLM calls) until all four items below have been run and read. Run All, then
fill in the decision cell at the bottom yourself — this notebook computes, it does not conclude.

| item | question | kills / resizes |
|---|---|---|
| P0.1 | does the router move once it can see corpus features it never had? | the whole relabelling program, if yes |
| P0.2 | does the spine reproduce on the 68% already banked? | the re-retrieval, if the spine itself is unstable |
| P0.3 | how many ceiling ties can *arithmetically* reach the certification bar? | the ceiling-tie leg of R1 |
| P0.4 | is `all_zero` really the higher-leverage population? | the ordering R1 runs in |

**Pre-registered readings** (write these down before running, so a result can't be reinterpreted
after the fact — see the plan's open question 2):

- **P0.1** — if the router's headroom captured or its margin against the per-lane best constant
  *improves materially* with `catalog_v3`'s corpus features wired in, the labelling program is
  premature: the bottleneck was a missing feature, not label quality. If it does **not** move,
  the corpus-feature explanation is ruled out and the labelling program is answering a live
  question.
- **P0.2** — mismatches should be near-zero on covered rows if the spine is stable. A large,
  systematic mismatch (not just `pure_rrf`, not explained by `fetch_limit`) means the spine
  itself doesn't reproduce, and re-retrieving the other 32% is not the fix.
- **P0.3** — if only a small share of ceiling ties can even in principle reach the ΔNDCG@10 ≥
  0.333 certification bar, the fake-tie leg of R1 is over-scoped relative to its yield.
- **P0.4** — this is arithmetic, not a live test: it restates why `all_zero` is the
  higher-leverage population and reproduces the pilot rescue rate directly from disk.

In [ ]:
from __future__ import annotations

import itertools
from pathlib import Path

import numpy as np
import pandas as pd

from composition.pool_v3 import CEILING, LabelledPool
from hybrid_search_rrf_dataset.labels import oracle_dir
from hybrid_search_rrf_dataset.router import (
    CATALOG_PATH,
    DATA_DIR,
    Representation,
    RouterExperiment,
)

ROUTES = ["dense_only", "pure_rrf", "sparse_only"]
SCORE_COLS = [f"score_{r}" for r in ROUTES]
pd.set_option("display.width", 140)

## P0.1 — retrain the router with the corpus channel it never had

`router.py:52` defaults `RouterExperiment` to `feature_table/catalog.parquet` — 75 columns,
**zero** corpus-relative features (identifier banks, length, stopword ratio, POS, markers only).
`data/v3/catalog_v3.parquet` has 118 columns: the same feature families plus ten corpus-relative
ones (`query_corpus.*`, `term_rarity.*`) — exactly the signal the rarity-gap finding named as
missing (71% of sparse wins feature-invisible). `RouterExperiment` already accepts `catalog_path`
as a constructor argument, so this is a one-line swap, not new code.

**Do not use `RouterExperiment.run()`'s own `best_const` / `headroom_captured` here.**
`_six_column` (`router.py:1006-1029`) sets `best_const = max(constants.values())` — the max of
three **global** means over the whole test set, not a per-lane choice. VERDICT's −0.0154 comparator
picks the best route **per lane**, then averages — since lanes disagree on which route wins,
per-lane-best is always ≥ global-best-of-means (Jensen's inequality), so the built-in number reads
artificially good. This cell refits the same estimator manually and computes the per-lane comparator
directly, matching `phase0_verify.ipynb`'s item 2 convention.

Caveat this doesn't fix: this is a single seeded 80/20 split, not VERDICT's lane-resampled bootstrap
estimator — read it as a fast, correctly-*directed* signal, not a replication of the precommitted CI.

In [ ]:
import warnings

from hybrid_search_rrf_dataset.router import StrategyRouter, _margin, _mean_objective

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)  # router.py:608, cosmetic

CATALOG_V3 = DATA_DIR / "v3" / "catalog_v3.parquet"


def per_lane_best_const(decisive: pd.DataFrame) -> float:
    """VERDICT's comparator: the best-scoring route PER LANE, then averaged —
    not router.py's own `_six_column.best_const`, which maxes three GLOBAL
    means and is therefore always <= this number whenever lanes disagree on
    which route wins."""
    per_lane = decisive.groupby("dataset")[SCORE_COLS].mean().idxmax(axis=1)
    total = sum(
        decisive.loc[decisive["dataset"] == lane, col].sum()
        for lane, col in per_lane.items()
    )
    return total / len(decisive)


rows = []
for label, catalog_path in [("shipped (0 corpus feats)", CATALOG_PATH),
                             ("catalog_v3 (+10 corpus feats)", CATALOG_V3)]:
    exp = RouterExperiment(catalog_path=catalog_path)
    data = exp.load()
    splits = {"random_within_lane": exp._split_random(data), "holdout_lane": exp._split_holdout(data)}
    for protocol, (train, test) in splits.items():
        train_model, train_tune = exp._prep_train(train)
        router = StrategyRouter(Representation.ENGINEERED, None).fit(train_model, all_rows="decisive")
        router.tune_thresholds(train_tune)
        decisive = test[_margin(test) >= router.decisive_margin]
        router_score = _mean_objective(decisive, router.predict_routes(decisive))
        weak_const = max(float(decisive[c].mean()) for c in SCORE_COLS)
        strong_const = per_lane_best_const(decisive)
        oracle = float(decisive[SCORE_COLS].max(axis=1).mean())
        rows.append({
            "catalog": label, "protocol": protocol, "n_test_decisive": len(decisive),
            "router": router_score, "oracle": oracle,
            "weak_const (router.py's own, wrong)": weak_const,
            "true_per_lane_const (VERDICT's comparator)": strong_const,
            "delta_vs_true_const": router_score - strong_const,
        })

p01 = pd.DataFrame(rows)
p01

**Read this against the pre-registered rule above.** The shipped router's measured delta against
the per-lane best constant was **−0.0154** pooled (VERDICT, lane-resampled estimator). Compare
`delta_vs_true_const` between the two catalog rows for the **same protocol** — `random_within_lane`
is the multi-lane split and the closer match to VERDICT's setup; `holdout_lane` has exactly one lane
in its test set, so its weak and true constants coincide by construction and it says nothing about
cross-lane averaging, only about generalization to one unseen corpus (`rarb-math`).

## P0.2 — does the spine reproduce on the 68% already banked?

Before buying ~58h of re-retrieval for the 72,163 uncovered rows: does re-deriving the spine from
the *already-persisted* rankings reproduce the shipped labels on the rows that have them? If not,
the instability isn't in the missing 32% — it's in the spine itself, and re-retrieval doesn't fix it.

`pool_v3.labels()` reads `v3/labels_rederived.parquet` in preference to `route_labels/labels.parquet`
when it exists — so this join is exactly what every v3 consumer already silently prefers.

In [ ]:
rederived = pd.read_parquet(DATA_DIR / "v3" / "labels_rederived.parquet").astype({"query_id": str})
shipped = pd.read_parquet(DATA_DIR / "route_labels" / "labels.parquet").astype({"query_id": str})

joined = shipped.merge(rederived, on=["dataset", "query_id"], suffixes=("_shipped", "_rederived"))
print(f"shared (dataset, query_id) rows: {len(joined):,}")
print()

for route in ROUTES:
    col = f"score_{route}"
    diff = (joined[f"{col}_shipped"] - joined[f"{col}_rederived"]).abs()
    mismatch = diff > 1e-9
    print(f"{route:12s}  mismatched: {mismatch.sum():6,} / {len(joined):,} "
          f"({mismatch.mean():.2%})   max |Δ|: {diff.max():.3f}")

winner_shipped = joined[[f"score_{r}_shipped" for r in ROUTES]].idxmax(axis=1).str.replace("_shipped", "")
winner_rederived = joined[[f"score_{r}_rederived" for r in ROUTES]].idxmax(axis=1).str.replace("_rederived", "")
flips = winner_shipped != winner_rederived
print(f"\nroute flips: {flips.sum():,} / {len(joined):,} ({flips.mean():.2%})")

**Plan's pre-registered tolerance:** dense/sparse mismatches ≤0.1% of rows, route flips ≤0.5%. If
`pure_rrf` is the outlier and the flip rate is small, suspect `fusion.py`'s `fetch_limit` default
(50) diverging from whatever depth the shipped labels were originally scored at — read the printed
numbers rather than assuming that's the cause.

## P0.3 — ceiling-tie yield, by arithmetic

At a ceiling tie, every route already ranks a judged-relevant doc **first** — `HitRate@1 = 1` for
all three, permanently. No amount of deepened judging can raise a rank-1 doc's relevance further,
so all resolvable signal lives in the `0.3·ΔNDCG@10` term, and `class_margin = 0.1` means
certification needs **ΔNDCG@10 ≥ 0.333** between the winning and losing route.

This cell computes, per ceiling-tie row, the **best-case** ΔNDCG@10 achievable between any two
routes by adding relevance judgments to currently-unjudged docs in the union of their top-10s —
never by removing an existing human judgment. A doc's relevance is one fact shared by every
route's score, so the search is over *which unjudged docs could be relevant*, not over per-route
grades. This is a true upper bound: if a row can't clear the bar even in the best case, no amount
of real judging will certify it either.

Scoped to the v2-origin tier (`route_labels/*_oracle/`, 46,142 keys) — the rankings the pipeline
already banked. The augmented rung has its own separate oracle layout and is out of scope here.

In [ ]:
def idcg(n: int, k: int = 10) -> float:
    n = min(n, k)
    return sum(1.0 / np.log2(r + 1) for r in range(1, n + 1))


def max_ndcg_gap(rankings: dict[str, list], relevant: set[str]) -> float:
    """Best-case NDCG@10(a) - NDCG@10(b) over any ordered route pair, from
    adding relevance judgments to currently-unjudged union docs only. For a
    fixed number of additions `m`, the numerator is linear in which docs are
    chosen, so the top-`m` docs by (weight_a - weight_b) are always optimal —
    iterating m from 0 to the full union and taking the max is exhaustive and
    exact, not a heuristic."""
    weights = {
        route: {doc: 1.0 / np.log2(i + 2) for i, doc in enumerate(docs)}
        for route, docs in rankings.items()
    }
    union = set().union(*rankings.values()) - relevant
    base_n = len(relevant)
    best = 0.0
    for a, b in itertools.permutations(rankings, 2):
        base_a = sum(weights[a].get(d, 0.0) for d in relevant)
        base_b = sum(weights[b].get(d, 0.0) for d in relevant)
        if base_n:
            best = max(best, (base_a - base_b) / idcg(base_n))
        ordered = sorted(
            union, key=lambda d: weights[a].get(d, 0.0) - weights[b].get(d, 0.0), reverse=True
        )
        cum_a = cum_b = 0.0
        for m, d in enumerate(ordered, start=1):
            cum_a += weights[a].get(d, 0.0)
            cum_b += weights[b].get(d, 0.0)
            best = max(best, ((base_a + cum_a) - (base_b + cum_b)) / idcg(base_n + m))
    return best

In [ ]:
pool = LabelledPool()
classified = pool.classify(pool.labels())
ceiling_ties = classified[
    (classified["kind"] == "genuine_tie")
    & (classified["oracle"] >= CEILING)
    & (~classified["native"])
].copy()
print(f"v2-origin ceiling ties in scope: {len(ceiling_ties):,}")

In [ ]:
qrels_cache: dict[str, pd.DataFrame] = {}
oracle_cache: dict[str, pd.DataFrame] = {}


def lane_qrels(lane: str) -> pd.DataFrame:
    if lane not in qrels_cache:
        qrels_cache[lane] = pd.read_parquet(
            DATA_DIR / lane / "qrels.parquet"
        ).astype({"query_id": str})
    return qrels_cache[lane]


def lane_oracle(lane: str) -> pd.DataFrame | None:
    if lane not in oracle_cache:
        path = oracle_dir(DATA_DIR / "route_labels", lane) / "rows.parquet"
        oracle_cache[lane] = (
            pd.read_parquet(path, columns=["query_id", "route_rankings"]).astype({"query_id": str})
            if path.exists() else None
        )
    return oracle_cache[lane]


gaps = []
skipped = 0
for row in ceiling_ties.itertuples():
    oracle_rows = lane_oracle(row.dataset)
    if oracle_rows is None:
        skipped += 1
        continue
    match = oracle_rows.loc[oracle_rows["query_id"] == str(row.query_id)]
    if match.empty:
        skipped += 1
        continue
    rankings = {r: list(v) for r, v in match.iloc[0]["route_rankings"].items()}
    qrels = lane_qrels(row.dataset)
    relevant = set(
        qrels.loc[
            (qrels["query_id"] == str(row.query_id)) & (qrels["relevance"] >= row.min_relevance),
            "doc_id",
        ].astype(str)
    )
    gaps.append(max_ndcg_gap(rankings, relevant))

gaps = pd.Series(gaps, dtype=float)
BAR = 0.333  # class_margin 0.1, lexicographic objective -> ΔNDCG@10 >= 0.1/0.3
print(f"rows scored: {len(gaps):,}  (skipped, no banked ranking: {skipped:,})")
print(f"share with achievable ΔNDCG@10 >= {BAR}: {(gaps >= BAR).mean():.1%}")
gaps.describe()

## P0.4 — confirm the retargeting: `all_zero` before ceiling ties

`HitRate@1` carries weight 0.7; `NDCG@10` carries 0.3. `all_zero` rows score zero on both terms —
a rescued row can move the *0.7-weighted* term from 0 to 1. Ceiling ties, per P0.3, can only ever
move the 0.3-weighted term. This cell states the mass argument in numbers and reproduces the pilot
rescue rate directly from the artifact on disk (not quoted from the pilot notebook's printed output).

In [ ]:
share_all_zero = (classified["kind"] == "all_zero").mean()
share_ceiling_tie = (
    (classified["kind"] == "genuine_tie") & (classified["oracle"] >= CEILING)
).mean()
print(f"all_zero:      {share_all_zero:.1%} of the pool — addressable via the 0.7-weighted Hit@1 term")
print(f"ceiling ties:  {share_ceiling_tie:.1%} of the pool — addressable only via the 0.3-weighted NDCG term")

In [ ]:
# n=2 stack_specific, one lane (crumb-legal-qa) — this is the pilot's full evidence,
# not a stand-in for a larger sample. Recomputed directly, not quoted.
stack_flag = pd.read_parquet(DATA_DIR / "legb_pilot" / "stack_flag.parquet")
az = stack_flag[stack_flag["kind_leg1"] == "all_zero"]
rescued = az[az["kind_legb"] != "all_zero"]
print(f"pilot (1 lane, n={len(stack_flag)}): all_zero rows = {len(az)}, "
      f"rescued by a stronger encoder alone = {len(rescued)} "
      f"({len(rescued) / max(len(az), 1):.1%})")
rescued["kind_legb"].value_counts() if len(rescued) else None

## Decision

Fill this in after reading the four results above — this notebook computes, it doesn't conclude.

| item | pre-registered reading | what I observed | verdict |
|---|---|---|---|
| P0.1 | router moves materially with corpus features -> program premature | | |
| P0.2 | mismatches near-zero on covered rows -> spine is stable | | |
| P0.3 | small certifiable share -> ceiling-tie leg is over-scoped | | |
| P0.4 | (arithmetic; confirms the ordering) | | |

**Overall: proceed as scoped / re-scope / stop**, and why.